In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail \
     --retry 999 \
     --retry-all-errors \
     --retry-delay 5 \
     --retry-max-time 600 \
     http://gateway:8001/api/games


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def find_agi_arc_repo() -> Path:
    candidates = []
    env_repo = os.getenv('AGI_ARC_REPO')
    if env_repo:
        candidates.append(Path(env_repo))
    candidates.extend([
        Path.cwd(),
        Path('/kaggle/working/agi-arc'),
    ])
    input_root = Path('/kaggle/input')
    if input_root.exists():
        candidates.extend(sorted(path for path in input_root.iterdir() if path.is_dir()))
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / 'src' / 'arc_agi3').exists() and (candidate / 'scripts' / 'kaggle_competition_submission.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the agi-arc repo in working or input. Clone the repo to /kaggle/working/agi-arc or attach it as a Kaggle Dataset input.')


if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    COMPETITION_ROOT = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3'
    REPO_SOURCE = find_agi_arc_repo()
    WORKING_COPY = '/kaggle/working/ARC-AGI-3-Agents-agi-arc-submission'
    os.environ['AGI_ARC_REPO'] = str(REPO_SOURCE)
    os.environ.setdefault('ARC_AGI3_MAX_STEPS', '128')
    os.environ.setdefault('ARC_AGI3_EXPLORE_STEPS', '24')
    os.environ.setdefault('ARC_AGI3_USE_COMPRESSARC', '1')
    os.environ.setdefault('ARC_AGI3_USE_ARCMDL', '1')
    print(f'Using agi-arc source: {REPO_SOURCE}')
    print(f'Writable execution copy will be created at: {WORKING_COPY}')
    subprocess.run(
        [
            sys.executable,
            str(REPO_SOURCE / 'scripts' / 'kaggle_competition_submission.py'),
            '--repo-root',
            str(REPO_SOURCE),
            '--competition-root',
            COMPETITION_ROOT,
            '--working-root',
            '/kaggle/working',
            '--description',
            'ARC-AGI-3-Agents-agi-arc-submission',
        ],
        check=True,
    )


In [ ]:
# Non-rerun mode: produce a dummy submission so the notebook
# has valid output for the initial commit / save.
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    pd.DataFrame({'task_id': ['dummy'], 'output': ['[[0]]']}).to_parquet(
        'submission.parquet', index=False)
    print('Non-competition mode: wrote dummy submission.parquet')
